# Notebook 1 — Supervised Baseline Ablation (2CH + 4CH, biplane EF)

Purpose: Produce the **supervised baseline rows** of the final ablation table, the *clinically correct* way.

What this notebook does:
- Loads the CAMUS dataset, **both 2CH and 4CH views** (4 images per patient: 2CH-ED, 2CH-ES, 4CH-ED, 4CH-ES).
- Trains a U-Net at **4 label fractions: 1%, 5%, 10%, 100%** with **3 seeds each**.
- Reports **per-view Dice and HD95** (2CH and 4CH separately) and **biplane Simpson's EF MAE** (the clinical standard).
- Saves CSV to `results/supervised_baseline.csv`.

Why both views: biplane EF is the standard clinical measurement. CAMUS was specifically designed to support it. Restricting to a single view systematically degrades EF accuracy by 2–4 percentage points and weakens the thesis defense.

Estimated runtime: ~5–6 h on an RTX 3060 (training set is now 1600 images at 100%).

In [1]:
# Cell 1 — Imports, device, paths, splits
import os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import SimpleITK as sitk
import cv2
import torchvision.transforms.functional as TF
from scipy.ndimage import label as cc_label
from medpy.metric.binary import hd95

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BASE_PATH = 'database/'
RESULTS_DIR = 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

all_patients = sorted([d for d in os.listdir(BASE_PATH)
                       if os.path.isdir(os.path.join(BASE_PATH, d))])
random.seed(42); random.shuffle(all_patients)
train_list = all_patients[:400]
val_list   = all_patients[400:450]
test_list  = all_patients[450:500]
print(f"Train/Val/Test patients: {len(train_list)}/{len(val_list)}/{len(test_list)}")

Device: cuda
Train/Val/Test patients: 400/50/50


In [2]:
# Cell 2 — Dataset: BOTH views, binary LV
class CamusDatasetFull(Dataset):
    """CAMUS dataset, both 2CH and 4CH views, binary LV.

    Each patient -> 4 samples. 400 train patients -> 1600 images.
    Each item returns (img, mask, view, phase, patient_id).
    """
    def __init__(self, patient_list, base_path, target_size=(256, 256), augment=False):
        self.base_path = base_path
        self.target_size = target_size
        self.augment = augment
        self.samples = []
        for p_id in patient_list:
            for view in ["2CH", "4CH"]:
                for phase in ["ED", "ES"]:
                    self.samples.append({
                        "id": p_id, "view": view, "phase": phase,
                        "img":  f"{p_id}_{view}_{phase}.nii.gz",
                        "mask": f"{p_id}_{view}_{phase}_gt.nii.gz",
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_path  = os.path.join(self.base_path, s["id"], s["img"])
        mask_path = os.path.join(self.base_path, s["id"], s["mask"])
        img  = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(img_path))).astype(np.float32)
        mask = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(mask_path))).astype(np.float32)
        mask = (mask == 1).astype(np.float32)
        img  = cv2.resize(img,  self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)
        img  = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img_t  = torch.from_numpy(img).unsqueeze(0).float()
        mask_t = torch.from_numpy(mask).long()
        if self.augment:
            if random.random() > 0.5:
                img_t  = TF.hflip(img_t)
                mask_t = TF.hflip(mask_t.unsqueeze(0)).squeeze(0)
            angle = random.uniform(-15, 15)
            img_t  = TF.rotate(img_t, angle)
            mask_t = TF.rotate(mask_t.unsqueeze(0), angle).squeeze(0)
        return img_t, mask_t, s["view"], s["phase"], s["id"]

train_ds_full = CamusDatasetFull(train_list, BASE_PATH, augment=True)
val_ds        = CamusDatasetFull(val_list,   BASE_PATH, augment=False)
test_ds       = CamusDatasetFull(test_list,  BASE_PATH, augment=False)
print(f"Dataset sizes — train: {len(train_ds_full)}, val: {len(val_ds)}, test: {len(test_ds)}")

Dataset sizes — train: 1600, val: 200, test: 200


In [3]:
# Cell 3 — U-Net + Combined Loss
def double_conv(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in,  c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )

class UNet(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.enc1 = double_conv(1,   64);  self.enc2 = double_conv(64,  128)
        self.enc3 = double_conv(128, 256); self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2); self.dec3 = double_conv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec2 = double_conv(256, 128)
        self.up1 = nn.ConvTranspose2d(128,  64, 2, stride=2); self.dec1 = double_conv(128,  64)
        self.final = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__(); self.ce = nn.CrossEntropyLoss()
    def forward(self, pred, target):
        ce = self.ce(pred, target)
        ps = F.softmax(pred, dim=1)
        t_oh = F.one_hot(target, 2).permute(0, 3, 1, 2).float()
        inter = (ps * t_oh).sum((2, 3)); union = (ps + t_oh).sum((2, 3))
        dice = (1 - (2 * inter + 1e-6) / (union + 1e-6)).mean()
        return 0.5 * ce + 0.5 * dice
print("Model + loss ready.")

Model + loss ready.


In [4]:
# Cell 4 — Metrics: per-view Dice/HD95 + BIPLANE Simpson's EF
SPACING = (0.3, 0.3)    # mm/pixel after resize (approximate, see thesis methodology)

def get_largest_cc(mask):
    if mask.sum() == 0: return mask
    lbl, n = cc_label(mask)
    if n == 0: return mask
    counts = np.bincount(lbl.flat)[1:]
    return (lbl == counts.argmax() + 1).astype(np.uint8)

def dice_score(p, t):
    p = (p > 0.5).astype(np.float32); t = (t > 0.5).astype(np.float32)
    return (2 * (p * t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

def _disk_diameters(mask, spacing, num_disks=20):
    """Return list of diameters (mm) along long axis and the disk height (mm)."""
    y_idx, _ = np.where(mask > 0.5)
    if len(y_idx) == 0:
        return None, None
    y_min, y_max = y_idx.min(), y_idx.max()
    h_px = y_max - y_min
    if h_px == 0:
        return None, None
    disk_h_mm = (h_px / num_disks) * spacing[0]
    diameters = []
    for i in range(num_disks):
        y = int(y_min + (i + 0.5) * (h_px / num_disks))
        w_px = int((mask[y, :] > 0.5).sum())
        diameters.append(w_px * spacing[1])
    return diameters, disk_h_mm

def biplane_simpson_volume(mask_2ch, mask_4ch, spacing=SPACING, num_disks=20):
    """Modified Simpson's biplane (method of disks). Returns mL.

    At each disk level, the LV cross-section is approximated as an ellipse with
    semi-axes from the 2CH and 4CH diameters:
        V = sum_i (pi/4) * d_2CH_i * d_4CH_i * h
    """
    d2, h2 = _disk_diameters(mask_2ch, spacing, num_disks)
    d4, h4 = _disk_diameters(mask_4ch, spacing, num_disks)
    if d2 is None or d4 is None:
        return 0.0
    h = 0.5 * (h2 + h4)
    vol_mm3 = sum((np.pi / 4) * a * b * h for a, b in zip(d2, d4))
    return vol_mm3 / 1000.0

@torch.no_grad()
def evaluate_full(model, dataset, device):
    """Compute per-view Dice/HD95 and biplane EF MAE."""
    model.eval()
    dices = {"2CH": [], "4CH": []}
    hds   = {"2CH": [], "4CH": []}
    # Index by patient -> view -> phase -> dataset idx
    per_pat = {}
    for k, s in enumerate(dataset.samples):
        per_pat.setdefault(s["id"], {}).setdefault(s["view"], {})[s["phase"]] = k

    ef_t_list, ef_p_list = [], []
    for p_id, views in per_pat.items():
        if not ("2CH" in views and "4CH" in views):
            continue
        if not all(ph in views[v] for v in ("2CH", "4CH") for ph in ("ED", "ES")):
            continue
        gts, preds = {}, {}
        for v in ("2CH", "4CH"):
            for ph in ("ED", "ES"):
                img, m, _, _, _ = dataset[views[v][ph]]
                pred = torch.argmax(model(img.unsqueeze(0).to(device)), dim=1).cpu().numpy()[0]
                pred = get_largest_cc(pred)
                gts[(v, ph)] = m.numpy(); preds[(v, ph)] = pred
                dices[v].append(dice_score(pred, gts[(v, ph)]))
                if pred.sum() > 0 and gts[(v, ph)].sum() > 0:
                    try: hds[v].append(hd95(pred, gts[(v, ph)]) * SPACING[0])
                    except Exception: pass
        edv_p = biplane_simpson_volume(preds[("2CH","ED")], preds[("4CH","ED")])
        esv_p = biplane_simpson_volume(preds[("2CH","ES")], preds[("4CH","ES")])
        edv_t = biplane_simpson_volume(gts[("2CH","ED")],   gts[("4CH","ED")])
        esv_t = biplane_simpson_volume(gts[("2CH","ES")],   gts[("4CH","ES")])
        if edv_p > 0 and edv_t > 0:
            ef_p_list.append((edv_p - esv_p) / edv_p * 100)
            ef_t_list.append((edv_t - esv_t) / edv_t * 100)
    return {
        "dice_2CH":  float(np.mean(dices["2CH"])),
        "dice_4CH":  float(np.mean(dices["4CH"])),
        "dice_mean": float(np.mean(dices["2CH"] + dices["4CH"])),
        "hd95_2CH_mm": float(np.mean(hds["2CH"])) if hds["2CH"] else np.nan,
        "hd95_4CH_mm": float(np.mean(hds["4CH"])) if hds["4CH"] else np.nan,
        "ef_mae_biplane": float(np.mean(np.abs(np.array(ef_t_list) - np.array(ef_p_list)))) if ef_p_list else np.nan,
        "n_patients_ef": len(ef_p_list),
    }
print("Metrics utilities defined (biplane).")

Metrics utilities defined (biplane).


In [5]:
# Cell 5 — Seeding, fraction subset, training loop
# OPTIMIZED: AMP (mixed precision) + grad clipping for the RTX 3060.
#            U-Net trains from scratch, so a single LR is kept (no LP-FT needed here).
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def fraction_indices(n_total, fraction, seed):
    rng = np.random.default_rng(seed)
    k = max(4, int(round(n_total * fraction)))
    return rng.choice(n_total, size=k, replace=False)

def quick_val_dice(model, val_ds, device, batch=8):
    model.eval(); d_acc, n = 0.0, 0
    with torch.no_grad():
        for imgs, msks, *_ in DataLoader(val_ds, batch_size=batch):
            preds = torch.argmax(model(imgs.to(device)), dim=1).cpu().numpy()
            for p, t in zip(preds, msks.numpy()):
                d_acc += dice_score(get_largest_cc(p), t); n += 1
    return d_acc / n

def train_unet(train_loader, val_ds, epochs, lr, device):
    model = UNet(n_classes=2).to(device)
    crit = CombinedLoss()
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler()   # ADDED: AMP (mixed precision)
    best_d, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        for imgs, msks, *_ in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():            # ADDED: AMP
                loss = crit(model(imgs), msks)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # ADDED: grad clipping (stability)
            scaler.step(opt); scaler.update()
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            avg = quick_val_dice(model, val_ds, device)
            if avg > best_d:
                best_d = avg
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ep {ep+1:>3}/{epochs} | val_dice={avg:.4f}")
    model.load_state_dict(best_state)
    return model, best_d
print("Training utilities defined.")

Training utilities defined.


In [6]:
# Cell 6 — MAIN EXPERIMENT LOOP
FRACTIONS = [0.01, 0.05, 0.10, 1.0]
SEEDS     = [42, 123, 7]
EPOCHS    = 50
LR        = 1e-4
BATCH     = 8

# ADDED: RTX 3060 acceleration knobs
torch.backends.cudnn.benchmark = True     # fixed 256x256 input -> faster convolutions
NUM_WORKERS = 0           # Windows/Jupyter: keep 0 (spawn re-imports the kernel and can hang).
                          # On Linux set 2-4. pin_memory + the cache below are the safe speedups.

# ADDED: cache decoded val/test frames so they are not re-read from .nii.gz every eval
class CachedDataset(torch.utils.data.Dataset):
    def __init__(self, base): self.base, self._cache = base, {}
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        if i not in self._cache: self._cache[i] = self.base[i]
        return self._cache[i]
    @property
    def samples(self): return self.base.samples   # evaluate_full() needs this

if not isinstance(val_ds, CachedDataset):  val_ds  = CachedDataset(val_ds)
if not isinstance(test_ds, CachedDataset): test_ds = CachedDataset(test_ds)

records = []; t0 = time.time()
for frac in FRACTIONS:
    for seed in SEEDS:
        run = f"supervised_frac{int(frac*100):03d}_seed{seed}"
        print(f"\n=== {run} ===")
        set_seed(seed)
        if frac < 1.0:
            sub = Subset(train_ds_full, fraction_indices(len(train_ds_full), frac, seed))
        else:
            sub = train_ds_full
        print(f"  train samples: {len(sub)}")
        # OPTIMIZED: pin_memory + (optional) workers for faster host->GPU loading
        loader = DataLoader(sub, batch_size=BATCH, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=(NUM_WORKERS > 0))
        model, best_val = train_unet(loader, val_ds, EPOCHS, LR, device)
        m = evaluate_full(model, test_ds, device)
        m.update({"run": run, "fraction": frac, "seed": seed,
                  "n_train": len(sub), "best_val_dice": best_val})
        records.append(m)
        print(f"  TEST  dice2CH={m['dice_2CH']:.4f}  dice4CH={m['dice_4CH']:.4f}  "
              f"hd95_2CH={m['hd95_2CH_mm']:.2f}mm  hd95_4CH={m['hd95_4CH_mm']:.2f}mm  "
              f"biplane_EF_MAE={m['ef_mae_biplane']:.2f}%")
        pd.DataFrame(records).to_csv(os.path.join(RESULTS_DIR, "supervised_baseline.csv"), index=False)
        torch.cuda.empty_cache()   # ADDED: free VRAM between the 12 runs (OOM safety)
print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")


=== supervised_frac001_seed42 ===
  train samples: 16


C:\Users\najib\AppData\Local\Temp\ipykernel_2240\1367017131.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()   # ADDED: AMP (mixed precision)
C:\Users\najib\AppData\Local\Temp\ipykernel_2240\1367017131.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():            # ADDED: AMP


  ep   5/50 | val_dice=0.1616
  ep  10/50 | val_dice=0.1675
  ep  15/50 | val_dice=0.1782
  ep  20/50 | val_dice=0.0375
  ep  25/50 | val_dice=0.0446
  ep  30/50 | val_dice=0.2776
  ep  35/50 | val_dice=0.7491
  ep  40/50 | val_dice=0.7482
  ep  45/50 | val_dice=0.7545
  ep  50/50 | val_dice=0.7808
  TEST  dice2CH=0.7811  dice4CH=0.7950  hd95_2CH=6.36mm  hd95_4CH=6.86mm  biplane_EF_MAE=35.19%

=== supervised_frac001_seed123 ===
  train samples: 16
  ep   5/50 | val_dice=0.1925
  ep  10/50 | val_dice=0.2648
  ep  15/50 | val_dice=0.0000
  ep  20/50 | val_dice=0.0000
  ep  25/50 | val_dice=0.1417
  ep  30/50 | val_dice=0.4439
  ep  35/50 | val_dice=0.6405
  ep  40/50 | val_dice=0.6272
  ep  45/50 | val_dice=0.7518
  ep  50/50 | val_dice=0.7905
  TEST  dice2CH=0.7818  dice4CH=0.7628  hd95_2CH=6.65mm  hd95_4CH=8.37mm  biplane_EF_MAE=122.38%

=== supervised_frac001_seed7 ===
  train samples: 16
  ep   5/50 | val_dice=0.0000
  ep  10/50 | val_dice=0.3617
  ep  15/50 | val_dice=0.0000
  ep  2

In [8]:
# Cell 7 — Aggregate (mean ± std across seeds)
df = pd.read_csv(os.path.join(RESULTS_DIR, "supervised_baseline.csv"))
metrics = ["dice_2CH","dice_4CH","dice_mean","hd95_2CH_mm","hd95_4CH_mm","ef_mae_biplane"]
rows = []
for frac, g in df.groupby("fraction"):
    r = {"fraction": frac}
    for m in metrics:
        r[f"{m}_mean"] = g[m].mean()
        r[f"{m}_std"]  = g[m].std()
    rows.append(r)
agg = pd.DataFrame(rows).sort_values("fraction")
agg.to_csv(os.path.join(RESULTS_DIR, "supervised_baseline_summary.csv"), index=False)
print(agg.to_string(index=False))
print("\nSaved: results/supervised_baseline.csv + ..._summary.csv")

 fraction  dice_2CH_mean  dice_2CH_std  dice_4CH_mean  dice_4CH_std  dice_mean_mean  dice_mean_std  hd95_2CH_mm_mean  hd95_2CH_mm_std  hd95_4CH_mm_mean  hd95_4CH_mm_std  ef_mae_biplane_mean  ef_mae_biplane_std
     0.01       0.775767      0.009855       0.773034      0.019023        0.774400       0.012733          6.388905         0.244808          7.759358         0.799344            60.663132           53.723128
     0.05       0.864517      0.007417       0.862393      0.002168        0.863455       0.004680          3.816741         0.219927          4.019026         0.209848            22.785058            7.223210
     0.10       0.893934      0.004712       0.888358      0.003552        0.891146       0.000751          3.082343         0.031092          3.341586         0.094222            13.858374            2.854204
     1.00       0.927035      0.000998       0.924762      0.003252        0.925898       0.002124          2.048115         0.057223          2.218629         